# Function define

In [ ]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

!pip install scikit-learn==1.2.2
import sklearn
sklearn.__version__

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
import pypfopt
import random
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy
from sklearn.metrics import davies_bouldin_score
from collections import Counter

pypfopt.__version__

!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

def modify_date(date):
    return date[:10]

def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.1/985.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 65.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.0
    Uninstalling scikit-learn-1.6.0:
      Successfully uninstalled scikit-learn-1.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.2.2 which is incompatible.
mlxtend 0.23.3 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.51
    Uninstalling yfinance-0.2.51:
      S

In [ ]:
def get_portfolio_PR(result,n_preselected_stock=40,n_seed=14,max_weight=0.14):
  # Preselection parameters set up
  #method = "K_means_multiple"# or avg or anyother method
  method = "K_means_multiple"
  stock_type = "S&P100"
  end_swan = "Swan8"         #equals the last swan before the swan we are going to predict
  start_swan="Swan4"
  start = dt(2019,1,1)
  end = dt(2020,2,15)
  selected_stock_number = n_preselected_stock     #number of stock preselected
  start_swan_date = dt(2020,2,15)
  end_swan_date = dt(2021,3,23)
  w1 = max_weight
  w2 = max_weight
  MA_days = 30
  N = 7 # This parameter is set to help classification method like K-means
  Min_appear = 2
  Seeds = n_seed

  #read all stocks and their swan beta
  if stock_type == "S&P100":
      portfolio_beta = pd.read_excel("All_SwanBeta100.xlsx",index_col=0)
      #portfolio_beta = pd.read_excel("BetaMatrix S&P100 Cathy(Yijue) 2022.xlsx",index_col=0)
  else:
      portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

  optdata = pd.read_excel("optdata_MVO2.xlsx",index_col=0,sheet_name = 1)
  portfolio_beta = portfolio_beta[portfolio_beta.index.isin(optdata.index)]

  selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
  portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

  portfolio_beta

  corr=portfolio_beta.corr()
  selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
  corr1={}
  for i in range(selected_corr.shape[1]):
      for j in range(i+1,selected_corr.shape[0]):
          selected_corr.iloc[j,i]=np.nan

  selected_corr

  avg=selected_corr.mean(axis=1)
  weight_swan = avg/sum(avg)
  selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
  #portfolio_beta["corr_beta"]=selected_beata@weight_swan
  #portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

  portfolio_beta

  if method == "gamma":
      portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
      portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
      portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
      preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
  elif method =="average":
      portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
      #Select 18 stocks
      portfolio_beta=portfolio_beta.sort_values(by=["avg"])
      preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

  elif method == "K_means":
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Y_predic = selected_beta[end_swan].copy()
      Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      labels = Kmean.labels_
      current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      print(current_dbscore)
      selected_beta["Group"]=Kmean.labels_
      print(selected_beta['Group'])
      res = ct.mean(axis=1)
      res [res<0]=0

      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group

  # Use results from multiple clusters
  elif method=="K_means_multiple":
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Y_predic = selected_beta[end_swan].copy()
      seed_dict={}

      # Run 1000 trials with davies bouldin index
      for r in range(1,1001):
        Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
        Kmean.fit(selected_beta)
        labels = Kmean.labels_
        # Evaluate the davies-bouldin score
        dbscore=metrics.davies_bouldin_score(selected_beta, labels)
        seed_dict[r]=dbscore
        sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

      # Put all stocks into a stock list
      stock_list=[]
      max_length=0
      for index in range(0,Seeds):
        selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
        Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
        Kmean.fit(selected_beta)
        ct = Kmean.cluster_centers_
        Kmean.labels_
        selected_beta["Group"]=Kmean.labels_
        res = ct.mean(axis=1)
        res [res<0]=0
        min_group = np.where(res==min(res))
        for i in range(N):
            Y_predic[selected_beta["Group"] == i] = res [i]
        selected_beta ["avg"]=Y_predic.values
        selected_beta=selected_beta.sort_values(by=["avg"])
        first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
        if first_group.shape[0] <10:
            second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
            preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
        else:
            preselected_stock = first_group
        if preselected_stock.shape[0]>max_length:
          max_length=preselected_stock.shape[0]
        stock_list.extend(preselected_stock.index.values)
      print(stock_list)

      # Count the times each stock appear
      values,counts=np.unique(stock_list,return_counts=True)
      count_dict={}
      preselected_stock=[]
      for t in range(0,len(values)):
        count_dict[values[t]]=counts[t]
      sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
      print(sorted_count)

      d = {'stock':[], 'frequency':[]}
      df = pd.DataFrame(data=d)
      for item in sorted_count:
        df.loc[len(df.index)] = [item[0],item[1]]
      stock_frequency_name = f"Astock_frequency_w1_{w1}_Seeds_{Seeds}.csv"
      df.to_csv(stock_frequency_name)

      for p in range(0,min(selected_stock_number,len(sorted_count))):
        if sorted_count[p][1]>=Min_appear:
          preselected_stock.append(sorted_count[p][0])

      preselected_stock = sorted(preselected_stock)
      selected_beta=selected_beta.loc[preselected_stock]
      selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
      #selected_beta["avg"]=selected_beta["avg"].mean()
      preselected_stock = selected_beta
      print(preselected_stock)

  elif method == "Custom":
      # Step 1: Calculate Davies-Bouldin index for random seeds
      selected_beta = portfolio_beta.loc[:, start_swan:end_swan].dropna()
      dbi_scores = []

      # Loop through random seeds from 1 to 1000 to find the best seeds with lowest DBI
      for R in range(1, 1001):
          kmeans = KMeans(n_clusters=N, random_state=R)
          labels = kmeans.fit_predict(selected_beta)
          dbi = davies_bouldin_score(selected_beta, labels)
          dbi_scores.append((R, dbi))

      # Sort by Davies-Bouldin index and select the top 5 seeds with lowest DBI
      top_seeds = sorted(dbi_scores, key=lambda x: x[1])[:Seeds]


      stock_counter = Counter()

      for R, _ in top_seeds:

          selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
          Y_predic = selected_beta[end_swan].copy()
          Kmean = KMeans(n_clusters=N,random_state=R)#set the random_state to keep modeling the same every run
          Kmean.fit(selected_beta)
          ct = Kmean.cluster_centers_
          Kmean.labels_
          selected_beta["Group"]=Kmean.labels_
          res = ct.mean(axis=1)
          res [res<0]=0

          min_group = np.where(res==min(res))
          for i in range(N):
              Y_predic[selected_beta["Group"] == i] = res [i]
          selected_beta ["avg"]=Y_predic.values
          selected_beta=selected_beta.sort_values(by=["avg"])
          first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
          if first_group.shape[0] <10:
              second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
              preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=True)
          else:
              preselected_stock = first_group

          stock_counter.update(preselected_stock.index)

      top_stocks = [stock for stock, count in stock_counter.most_common(22)]

  ct

  res

  preselected_stock.info()

  #order = list(preselected_stock.index)
  #random.shuffle(order)
  #preselected_stock = preselected_stock.reindex(index = order)
  preselected_stock

  #holt step
  portfolio_beta = pd.read_csv("HoltestimateS9.csv",index_col=0)
  if 'Ticker' in preselected_stock.columns:
      preselected_stock.set_index('Ticker', inplace=True)
  preselected_stock['avg'] = preselected_stock.index.map(portfolio_beta['Forecasted Swan9'])

  #using average step
  '''
  preselected_stock

  preselected_stock['avg'] = (preselected_stock['Swan4'] + preselected_stock['Swan5']+preselected_stock['Swan6']+preselected_stock['Swan7']+preselected_stock['Swan8'])/5
  '''

  preselected_stock["PR"]=1-preselected_stock["avg"]
  preselected_stock.to_csv('preselected_stocks.csv')
  preselected_stock

  """# Step 2 - Download data and adjuste price based on Swan Beta"""

  #next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
  #selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
  #swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

  #Set the paramter to download tradeinfo from Yahoo Finance
  selected_stock=preselected_stock
  selected_stock_list=selected_stock.index.values
  swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
  print(selected_stock_list)



  #download data from yahoo finance
  yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]

  yahooDataFrame.to_csv('yahooDataFrame.csv')
  yahooDataFrame=yahooDataFrame[selected_stock_list]
  yahooDataFrame

  #set parameters to calculate predict market drop
  market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
  #swan_to_predict=100
  total_day=yahooDataFrame.shape[0]
  t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
  t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
  #t2=total_day-t1
  t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
  dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
  t0=int(total_day-t1-t2)

  #calculate predict market drop
  md = [] #Make market drop data list named md


  j = 0 #Start from 0
  inc = dd/(t1-1) #Calculate unit increasing market drop
  for i in range(t1): #Append market drop data for t1
      md.append(round(j,4))
      j = j + inc

  base = (t2+1)**(1/(dd*100)) #Calculate log base
  l1 = [] #Make decreasing market drop list
  for i in range(t2):
      l1.append(math.log((i+2),base)/100)

  for i in range(t2): #Append market drop data for t2
      md.append(round((dd-l1[i]),4))

  for i in range(t0):
      md.append(0)

  market_drop = pd.DataFrame(md) #Change list into dataframe
  #market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
  market_drop1 = pd.read_csv("trainingMarketdrop.csv")

  market_drop.to_csv("market_drop.csv")

  market_drop

  swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
  yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

  yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
  selected_stock_list=yahooDataFrame.columns.values
  selected_stock_number = yahooDataFrame.shape[1]
  selected_stock_list

  yahooDataFrame.to_csv('yahooDataFrame.csv')
  yahooDataFrame

  """# Step3 - MVO"""

  #Set the paramter for MVO process
  Portfolio_Date=end

  #data cleaning - filling missing data in yahooDataFrame
  data_cleaning(yahooDataFrame)

  #calculated rf
  #bond_yield = pd.read_csv("bond_yield.csv", index_col=0)
  #bond_yield=np.log(bond_yield/100+1)
  #date=list(map(str,bond_yield.index.values))
  #date=list(map(modify_date,date))
  #week=pd.to_datetime(date).strftime("%W").values
  #week_1=np.roll(week,1)
  #week_difference=np.where(week!=week_1)
  #week_difference=np.delete(week_difference,0,None)
  #rf_week=bond_yield.iloc[week_difference-1]

  # Calculated R for 1 year and 0.5 year
  today_price = yahooDataFrame.iloc[-MA_days:,:]
  one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
  one_year_ago_date = one_year_ago_data.index.values[-1]
  half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
  half_year_ago_date = half_year_ago_data.index.values[-1]
  one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
  one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
  half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
  half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

  R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
  R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)

  # Calculate MA
  MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
  MA_Data=MA_Data.loc[one_year_ago_date:,:]
  MA_now = MA_Data.iloc[-1,:]
  MA_one_year_ago = MA_Data.iloc[0,:]
  MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

  # Calculate daily return
  daily_return = yahooDataFrame.pct_change().dropna()
  log_daily_return = np.log(1+daily_return)

  # Get week end stock price data
  stock_date = yahooDataFrame.index.values
  week=pd.to_datetime(stock_date).strftime("%W").values
  week_1=np.roll(week,1)
  week_difference=np.where(week!=week_1)
  week_difference=np.delete(week_difference,0,None)
  # Calculate weekly return

  weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
  log_weekly_return = np.log(1+weekly_return)

  # Get month end stock price data
  stock_date = yahooDataFrame.index.values
  month=pd.to_datetime(stock_date).strftime("%m").values
  month_1=np.roll(month,1)
  month_difference=np.where(month!=month_1)
  month_difference=np.delete(month_difference,0,None)
  # Calculate monthly return
  monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
  log_monthly_return = np.log(1+monthly_return)

  # Get semi-month end stock price data
  semi_month = []
  for i in range(1,len(month_difference)):
      semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
      semi_month.append(month_difference[i]-1)
  # Calculate semi-monthly return
  semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
  log_semi_monthly_return = np.log(1+semi_monthly_return)

  # Calculated Max Dropdown for each stock
  stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
  mdd=[]
  for column in stock_return_one_year.columns:
      mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
  mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
  mar=R_mv_one_year/mdd.iloc[:,0]

  #calculate weekly sd for each stock
  one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
  weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

  #Calculate garch_std sd for each stock
  one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
  res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
  judge = one_year_monthly_return.shape[0]
  for stock in one_year_monthly_return.columns:
      res=res_all[stock]
      h= het_arch(res)
      if judge < 100 or h ==0:
          garch_std = weekly_std

      else:#build Garch Model
          std = []
          if judge <=200:
              p = 0
              q = 6
          else:
              p = 1
              q = 1

          std.append(fit_garch(one_year_monthly_return[stock],p,q))
          garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

  #Calculate monthly sd for each stock
  one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
  monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

  #Calculate semi_monthly sd for each stock
  one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
  semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);

  # Calculated MaxDrop
  DD_range = 90
  total_range = 90
  DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
  DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
  max_DD=[-1]*selected_stock_number


  if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
      DD_start = DD_start.index[-1]
      DD_end = DD_end.index[-1]
      while DD_start > DD_end :
          DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
          DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
          if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
              DD_start = DD_start.index[-1]
              DD_mid = DD_mid.index[-1]
              price = yahooDataFrame.loc[DD_start:DD_mid,:]
              for i in range(len(yahooDataFrame.columns)):
                  max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
              total_range = total_range + 1
          else:
              break

  # Read OPT-data
  optdata = pd.read_excel("optdata_MVO2.xlsx",index_col=0,sheet_name = 1)
  P_current = MA_now
  P_fwd = optdata.P_fwd.loc[P_current.index]
  R_fwd=P_fwd/P_current-1
  R_60 = P_current/MA_one_year_ago - 1
  R_60_half = P_current/MA_half_year_ago - 1
  R_MA_1ago = R_mv_one_year
  R_MA_halfago = R_mv_half_year
  Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
  for i in R_fwd.index:
      if R_fwd[i] < 0.5*R_60[i]:
          Rfwd_adjusted[i] = R_60_half[i]
      elif R_fwd[i] > 1.3*R_60[i]:
          if 1.3*R_60[i] > 0.7*R_fwd[i]:
              Rfwd_adjusted[i] = 1.3*R_60[i]
          else:
              Rfwd_adjusted[i] = 0.7*R_fwd[i]
      else:
          Rfwd_adjusted[i] = R_fwd[i]
  Rfwd_adjusted[Rfwd_adjusted<0]=0

  price_now = today_price.iloc[-1,:]
  optimizer = optdata.loc[P_current.index,:]
  FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
  preselected_stock = preselected_stock.loc[selected_stock_list,:]
  preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
  preselected_stock["std_f"]=weekly_std.values
  preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
  preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
  preselected_stock.to_csv("preselected_Stocks_info.csv")
  p=FFC.MA_now
  EPS_fwd=optimizer.EPS_fwd
  EPS_ttm=optimizer.EPS_ttm
  fall=FFC.Fall_MDD
  FFC.to_csv("WEDataBank.csv")

  PE=p/EPS_fwd
  PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
  PEG[PEG<0]=2000

  cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
  E_2 = Rfwd_adjusted/fall
  E_3 = Rfwd_adjusted/PEG
  E_4 = Rfwd_adjusted
  std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)

  """# Optimization"""

  w0 = 1/selected_stock_number*np.ones(selected_stock_number)
  uplim_ini = np.zeros(selected_stock_number)
  uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
  for i in uplim3.index:
      if i in ["SPY","IWD"]:
          uplim3.loc[i,:]=0
  low_lim =  np.zeros(selected_stock_number)
  multi=lambda x: x.T @cov@x

  ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
  #ef.add_constraint(lambda w: w@E_2== 12)
  raw_weights = ef.max_sharpe(0.01)
  cleaned_weights = ef.clean_weights()
  print(cleaned_weights)
  ef.portfolio_performance(verbose=True)

  weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

  sd1 = np.sqrt(weight1@cov@weight1.T)
  R1 = weight1.T@E_2/100
  sharpe_ratio1 = (R1-0.01)/sd1
  print("new_sharpe:",sharpe_ratio1)

  """weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
  weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

  sd2 = np.sqrt(weight2@cov@weight2.T)
  R2 = weight2.T@E_2/100
  sharpe_ratio2 = (R2-0.01)/sd2
  print("old_sharpe:",sharpe_ratio2)
  """

  weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
  weight = weight[weight.weight >=0.030]
  weight.weight = weight.weight/sum(weight.weight)
  weight.to_csv("weights.csv")

  """See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference

  """

  if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

  elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

  stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

  stock_price.to_csv("tradeinfo.csv")
  stock_price=stock_price[weight.index]
  stock_price

  weight

  start=stock_price.loc["2020-02-19"]
  end = stock_price.loc["2020-03-23"]
  stock_drop=abs(end/start-1)
  market_drop = abs(OEX.loc["2020-03-23"]/OEX.loc["2020-02-19"]-1)

  beta = stock_drop/market_drop
  1-beta.T@weight

  beta

  stock_drop

  rest=1
  extra=0
  for i, row in weight.iterrows():
    if row['weight']>0.17:
      rest=rest-row['weight']
      extra=extra+(row['weight']-0.17)
      # row['weight']=0.17

  for i,row in weight.iterrows():
    if row['weight']<0.165:
      # row['weight']=row['weight']+row['weight']/rest*extra
      a=1


  weights_name = f"Aweights_w1_{w1}_Seeds_{Seeds}.csv"
  weight

  portfolio_beta = portfolio_beta.loc[weight.index]

  swan_beta_name = f"A_forcast_and_actual_Beta{w1}_Seeds_{Seeds}.csv"

  portfolio_beta

  a=stock_price@weight
  b=1-a.loc["2020-03-23"]/a.loc["2020-02-19"]
  1-b/market_drop

  injected_result = pd.DataFrame(index = stock_price.index)
  S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T

  portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
  S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
  injected_result["portfolio_value"]=stock_price@S1.T
  S2 = 10000/float(OEX[0])
  injected_result["oex"]=S2*OEX

  temp=injected_result.loc[["2020-02-19","2020-03-23"]]
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)
  PR=1-(Portfolio_DD/DD)
  print(PR)
  temp

  result.loc[n_seed,max_weight]=PR*100
  print(selected_stock_number,max_weight)

# Set up tests

In [ ]:
n_seeds = np.arange(16,20,1)
max_weights = np.arange(0.1,0.13,0.01)

# Create a DataFrame with specified index and columns, initializing with zeros
result = pd.DataFrame(0, index=n_seeds, columns=max_weights)
for n_seed in n_seeds:
  for max_weight in max_weights:
    get_portfolio_PR(result,n_seed=n_seed,max_weight=max_weight)
# Display the DataFrame

<ipython-input-13-fc98073146cc>:122: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-13-fc98073146cc>:122: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-13-fc98073146cc>:122: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selecte

['SO', 'ABT', 'AMGN', 'PFE', 'MDLZ', 'MRK', 'JNJ', 'CL', 'PEP', 'SBUX', 'PG', 'LLY', 'NEE', 'EXC', 'WMT', 'VZ', 'KO', 'SO', 'ABT', 'SBUX', 'MDLZ', 'MRK', 'CL', 'PEP', 'LLY', 'JNJ', 'KO', 'NEE', 'PG', 'VZ', 'EXC', 'AMZN', 'WMT', 'SO', 'ABT', 'SBUX', 'MDLZ', 'MRK', 'CL', 'PEP', 'LLY', 'JNJ', 'KO', 'NEE', 'PG', 'VZ', 'EXC', 'AMZN', 'WMT', 'SO', 'ABT', 'SBUX', 'MDLZ', 'MRK', 'JNJ', 'PEP', 'LLY', 'CL', 'KO', 'NEE', 'PG', 'WMT', 'VZ', 'EXC', 'SO', 'AMZN', 'ABT', 'AMGN', 'PFE', 'MDLZ', 'MRK', 'SBUX', 'JNJ', 'CL', 'COST', 'EXC', 'LLY', 'PG', 'KO', 'MO', 'WMT', 'PEP', 'VZ', 'NEE', 'SO', 'AMZN', 'ABT', 'AMGN', 'PFE', 'MDLZ', 'MRK', 'SBUX', 'JNJ', 'CL', 'COST', 'EXC', 'LLY', 'PG', 'KO', 'MO', 'WMT', 'PEP', 'VZ', 'NEE', 'SO', 'AMGN', 'COST', 'PFE', 'SBUX', 'MDLZ', 'MRK', 'SPG', 'JNJ', 'CL', 'BRK-B', 'ABT', 'PEP', 'LLY', 'PG', 'KO', 'MO', 'WMT', 'EXC', 'VZ', 'NEE', 'SO', 'ABT', 'SBUX', 'MDLZ', 'MRK', 'CL', 'PEP', 'LLY', 'JNJ', 'KO', 'NEE', 'PG', 'VZ', 'EXC', 'AMZN', 'WMT', 'SO', 'SBUX', 'ABT', 'AMG

[*********************100%%**********************]  20 of 20 completed
/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
 

OrderedDict([('ABT', 0.0), ('AMGN', 0.0), ('AMZN', 0.1), ('CL', 0.0), ('COST', 0.1), ('EXC', 0.04795), ('JNJ', 0.0), ('KO', 0.07511), ('LLY', 0.1), ('MDLZ', 0.0), ('MO', 0.0), ('MRK', 0.1), ('NEE', 0.1), ('PEP', 0.07131), ('PFE', 0.0), ('PG', 0.07739), ('SBUX', 0.0), ('SO', 0.1), ('VZ', 0.02824), ('WMT', 0.1)])
Expected annual return: 17.6%
Annual volatility: 8.4%
Sharpe Ratio: 1.98
new_sharpe: 1.982083565262589


[*********************100%%**********************]  11 of 11 completed
<ipython-input-13-fc98073146cc>:595: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
<ipython-input-13-fc98073146cc>:599: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
<ipython-input-13-fc98073146cc>:600: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'

...

[*********************100%%**********************]  22 of 22 completed
/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
 

OrderedDict([('ABT', 0.0), ('AMGN', 0.0), ('AMZN', 0.12), ('BRK-B', 0.0), ('CL', 0.0), ('COST', 0.10461), ('EXC', 0.0), ('JNJ', 0.0), ('KO', 0.04538), ('LLY', 0.12), ('MDLZ', 0.0), ('MO', 0.0), ('MRK', 0.08248), ('NEE', 0.12), ('PEP', 0.077), ('PFE', 0.0), ('PG', 0.04008), ('SBUX', 0.0), ('SO', 0.12), ('SPG', 0.05044), ('VZ', 0.0), ('WMT', 0.12)])
Expected annual return: 20.4%
Annual volatility: 8.1%
Sharpe Ratio: 2.39
new_sharpe: 2.390850208945553


[*********************100%%**********************]  11 of 11 completed


0.2829849004199687


<ipython-input-13-fc98073146cc>:595: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
<ipython-input-13-fc98073146cc>:599: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
<ipython-input-13-fc98073146cc>:600: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)


In [ ]:
result

,0.10,0.11,0.12
16,0.303311,0.344914,0.354521
17,0.246582,0.278976,0.282985
18,0.246582,0.278976,0.282985
19,0.246582,0.278976,0.282985
